# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umarfarukh786/FlyRank-task1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

This notebook treats the task as supervised binary ranking: identify which eligible content items have the observed decline label `is_declining_label = 1`. I will compare a readable Logistic Regression model, a shallow Decision Tree, and a bounded Random Forest. The models output probabilities so the evaluation matches the editorial workflow: pages are ranked for review rather than only assigned a hard class. Logistic Regression is the simplest interpretable reference; the tree can capture a few nonlinear thresholds; the forest is included only as a controlled complexity check. I will not use `trend_direction`, `trend_pct`, `content_id`, or `client_id` as features because the first two define the label and the IDs are grouping keys.

In [1]:
chosen_methods = ["logistic_regression", "decision_tree", "random_forest"]
print("Chosen methods:", ", ".join(chosen_methods))
print("Primary ranking metric: Precision@50; secondary metrics: ROC AUC and average precision")

Chosen methods: logistic_regression, decision_tree, random_forest
Primary ranking metric: Precision@50; secondary metrics: ROC AUC and average precision


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES

FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
BASELINE_PATH = ROOT / "data" / "processed" / "baseline_refresh_queue.csv"
RANDOM_STATE = 42

if not FEATURE_PATH.exists() or not BASELINE_PATH.exists():
    print("Prepared inputs not found; building the feature vector and baseline first.")
    for script_name in ("01_prepare_features.py", "02_baseline_score.py"):
        subprocess.run(
            [sys.executable, str(ROOT / "scripts" / script_name)],
            cwd=ROOT,
            check=True,
        )

frame = pd.read_csv(FEATURE_PATH)
baseline_frame = pd.read_csv(BASELINE_PATH)
if frame.empty:
    raise ValueError("The prepared feature vector is empty. Run the preparation pipeline first.")

required_columns = {"content_id", "client_id", "is_declining_label"}
missing_columns = required_columns.difference(frame.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

numeric_features = [column for column in MODEL_NUMERIC_FEATURES if column in frame.columns]
categorical_features = [column for column in MODEL_CATEGORICAL_FEATURES if column in frame.columns]
forbidden_features = {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"}
used_features = set(numeric_features + categorical_features)
leakage_columns = used_features.intersection(forbidden_features)
if leakage_columns:
    raise AssertionError(f"Forbidden columns found in model features: {sorted(leakage_columns)}")

target = frame["is_declining_label"].astype(int)
print(f"Eligible rows: {len(frame):,}")
print(f"Clients: {frame['client_id'].nunique():,}")
print(f"Observed decline base rate: {target.mean():.3f}")
print(f"Numeric features: {len(numeric_features)} | categorical features: {len(categorical_features)}")
print("Leakage check: passed")

Eligible rows: 30,000
Clients: 32
Observed decline base rate: 0.542
Numeric features: 18 | categorical features: 8
Leakage check: passed


## 3. Train + compare vs my baseline

The baseline is the existing deterministic `baseline_refresh_score` from Week 4. I evaluate its scores and every model probability on the same held-out rows. Precision@K is the main decision metric because a reviewer usually has a limited queue; ROC AUC and average precision provide broader ranking context, while precision, recall, and F1 describe the thresholded 0.5 classification view. A model is useful only if it improves the same holdout comparison without relying on forbidden label-derived columns.

In [7]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k

FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
BASELINE_PATH = ROOT / "data" / "processed" / "baseline_refresh_queue.csv"
RANDOM_STATE = 42

frame = pd.read_csv(FEATURE_PATH)
baseline_frame = pd.read_csv(BASELINE_PATH)
if frame.empty:
    raise ValueError("The prepared feature vector is empty. Run the preparation pipeline first.")

numeric_features = [column for column in MODEL_NUMERIC_FEATURES if column in frame.columns]
categorical_features = [column for column in MODEL_CATEGORICAL_FEATURES if column in frame.columns]
numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = frame[categorical_features].fillna("unknown").astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=categorical_features, dtype=float)
feature_frame = pd.concat([numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)], axis=1)
target = frame["is_declining_label"].astype(int)

all_indices = np.arange(len(frame))
clients = frame["client_id"].fillna("unknown").astype(str)
unique_clients = clients.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
test_clients = set(rng.permutation(unique_clients)[:max(1, int(round(len(unique_clients) * 0.20)))])
test_mask = clients.isin(test_clients).to_numpy()
train_indices = all_indices[~test_mask]
test_indices = all_indices[test_mask]
split_strategy = "client_holdout"
if (
    len(train_indices) == 0
    or len(test_indices) == 0
    or target.iloc[train_indices].nunique() < 2
    or target.iloc[test_indices].nunique() < 2
):
    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=target,
    )
    split_strategy = "stratified_row_holdout_fallback"

X_train = feature_frame.iloc[train_indices]
X_test = feature_frame.iloc[test_indices]
y_train = target.iloc[train_indices]
y_test = target.iloc[test_indices]

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

def metric_row(name, scores):
    hard_predictions = (scores >= 0.5).astype(int)
    return {
        "method": name,
        "roc_auc": roc_auc_score(y_test, scores),
        "average_precision": average_precision_score(y_test, scores),
        "precision_at_20": precision_at_k(y_test, scores, 20),
        "precision_at_50": precision_at_k(y_test, scores, 50),
        "precision_at_100": precision_at_k(y_test, scores, 100),
        "precision": precision_score(y_test, hard_predictions, zero_division=0),
        "recall": recall_score(y_test, hard_predictions, zero_division=0),
        "f1": f1_score(y_test, hard_predictions, zero_division=0),
    }

baseline_lookup = baseline_frame.set_index("content_id")["baseline_refresh_score"]
baseline_scores = frame.iloc[test_indices]["content_id"].map(baseline_lookup).fillna(0).to_numpy()
results = [metric_row("baseline_rules", baseline_scores)]
probabilities = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probabilities[name] = model.predict_proba(X_test)[:, 1]
    results.append(metric_row(name, probabilities[name]))

comparison = pd.DataFrame(results).set_index("method")
print(f"Rows: {len(frame):,} | train: {len(train_indices):,} | test: {len(test_indices):,}")
print(f"Split: {split_strategy} | held-out clients: {len(test_clients):,} | test base rate: {y_test.mean():.3f}")
display(comparison.round(3))

Rows: 30,000 | train: 27,675 | test: 2,325
Split: client_holdout | held-out clients: 6 | test base rate: 0.391


,roc_auc,average_precision,precision_at_20,precision_at_50,precision_at_100,precision,recall,f1
method,,,,,,,,
baseline_rules,0.627,0.468,0.15,0.24,0.36,0.499,0.189,0.274
logistic_regression,0.700,0.522,0.35,0.40,0.44,0.566,0.567,0.566
decision_tree,0.742,0.575,0.55,0.62,0.60,0.569,0.716,0.634
random_forest,0.747,0.610,0.70,0.68,0.70,0.560,0.741,0.638


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
best_method = comparison["precision_at_50"].idxmax()
best_scores = baseline_scores if best_method == "baseline_rules" else probabilities[best_method]

error_frame = frame.iloc[test_indices][["content_id", "client_id", "is_declining_label", "impressions_90d", "content_age_days", "avg_position", "ctr"]].copy()
error_frame["score"] = best_scores
error_frame["predicted_label"] = (best_scores >= 0.5).astype(int)
error_frame["error_type"] = np.select(
    [
        (error_frame["is_declining_label"] == 1) & (error_frame["predicted_label"] == 0),
        (error_frame["is_declining_label"] == 0) & (error_frame["predicted_label"] == 1),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)
print(f"Selected by Precision@50: {best_method}")
print("Error counts:")
display(error_frame["error_type"].value_counts().to_frame("rows"))
print("Three illustrative mistakes:")
display(
    error_frame[error_frame["error_type"] != "correct"]
    .sort_values("score", ascending=False)
    .head(3)
)

if best_method != "baseline_rules":
    fitted_model = models[best_method]
    if best_method == "logistic_regression":
        raw_importance = np.abs(fitted_model.named_steps["model"].coef_[0])
    else:
        raw_importance = fitted_model.feature_importances_
    importance = pd.DataFrame({"feature": feature_frame.columns, "importance": raw_importance})
    importance = importance.sort_values("importance", ascending=False).head(10)
    print("Top ten model features:")
    display(importance)

print(
    "Interpretation: these are observed associations in a client-held-out starter slice, "
    "not causal effects. False positives are pages the model ranks for review that are not "
    "declining under this proxy label; false negatives are declining pages it would miss. "
    "The final choice should balance Precision@50 against the operational cost of missed pages."
)

Selected by Precision@50: random_forest
Error counts:


,rows
error_type,
correct,1561
false_positive,529
false_negative,235


Three illustrative mistakes:


,content_id,client_id,is_declining_label,impressions_90d,content_age_days,avg_position,ctr,score,predicted_label,error_type
23250,content_d2dffcc697a4,client_f74efabef1,0,5091,144,14.1,0.20,0.737431,1,false_positive
23559,content_00603b0349b4,client_f74efabef1,0,1076,125,25.6,0.09,0.735212,1,false_positive
23750,content_e55b8ab078b0,client_f74efabef1,0,369,112,21.8,0.00,0.733797,1,false_positive


Top ten model features:


,feature,importance
5,log_impressions_90d,0.131904
9,days_with_impressions,0.130438
14,avg_position,0.111096
11,content_age_days,0.090817
32,age_tier_365+,0.037600
3,word_count,0.036792
6,log_clicks_90d,0.036497
4,char_count,0.036267
10,days_with_sessions,0.035031
13,ctr,0.034295


Interpretation: these are observed associations in a client-held-out starter slice, not causal effects. False positives are pages the model ranks for review that are not declining under this proxy label; false negatives are declining pages it would miss. The final choice should balance Precision@50 against the operational cost of missed pages.


## Self-check

Before submitting, rerun the notebook manually and confirm:

- [ ] Every section contains both reasoning and supporting code.
- [ ] The notebook runs top to bottom with no errors.
- [ ] The baseline and models use the same held-out rows and the same ranking metrics.
- [ ] The split is client-held-out when both classes are available; any fallback is printed.
- [ ] `trend_direction`, `trend_pct`, IDs, and any product decision flags are excluded from features.
- [ ] The comparison table includes the test base rate and Precision@20, @50, and @100.
- [ ] The error table shows false positives and false negatives, not only aggregate scores.
- [ ] Claims say observed, measured, or directional and stay limited to the anonymized starter slice.
- [ ] No client names, URLs, private queries, or credentials appear in outputs.
- [ ] The completed notebook is committed under `work/notebooks/w05_model.ipynb`.